In [1]:
import os
import re
import json
import time
import random
import requests
import pandas as pd
from urllib.parse import quote

In [1]:
class RedditLaptopScraper:
    def __init__(self, output_dir="reddit_dataset", min_words=5, delay=(1, 2)):
        self.output_dir = output_dir
        self.min_words = min_words
        self.delay = delay

        os.makedirs(self.output_dir, exist_ok=True)

        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36"
        }

        self.stats = {
            "saved_rows": 0,
            "saved_files": 0,
            "failed_models": 0
        }

    def fetch_url(self, url):
        try:
            response = requests.get(url, headers=self.headers, timeout=20)
            if response.status_code == 200:
                return response.text
            else:
                print(f"Failed: {response.status_code} -> {url}")
                return None
        except Exception as e:
            print(f"Error fetching {url}: {e}")
            return None

    def clean_text(self, text):
        if not text:
            return ""

        text = str(text)

        text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
        text = text.replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">")
        text = re.sub(r"\[.*?\]\(.*?\)", " ", text)
        text = re.sub(r"[^A-Za-z0-9\s.,!?;:()'\"/%+-]", " ", text)
        text = re.sub(r"([!?.,])\1+", r"\1", text)
        text = re.sub(r"\s+", " ", text)

        return text.strip()

    def normalize_text(self, text):
        if not text:
            return ""
        text = str(text).lower().strip()
        text = re.sub(r"\s+", " ", text)
        return text

    def count_words(self, text):
        if not text:
            return 0
        return len(str(text).split())

    def safe_filename(self, name):
        name = re.sub(r'[\\/*?:"<>|]', "_", name)
        name = re.sub(r"\s+", "_", name.strip())
        return name

    def get_query_variants(self, laptop_name):
        variants = [laptop_name]

        words = laptop_name.split()

        # shorter fallback queries
        if len(words) >= 3:
            variants.append(" ".join(words[:3]))   # e.g. ASUS VivoBook 15
            variants.append(" ".join(words[:4]))   # e.g. ASUS ZenBook 14 OLED

        # model code only query
        model_codes = re.findall(r"[A-Z]{1,5}\d{2,}[A-Z0-9]*", laptop_name.upper())
        variants.extend(model_codes)

        # remove duplicates, preserve order
        seen = set()
        final_variants = []
        for q in variants:
            q = q.strip()
            if q and q.lower() not in seen:
                final_variants.append(q)
                seen.add(q.lower())

        return final_variants

    def tag_topics(self, text):
        text = str(text).lower()

        return {
            "mentions_battery": int(any(word in text for word in ["battery", "backup", "battery life", "charging", "charger"])),
            "mentions_display": int(any(word in text for word in ["display", "screen", "oled", "brightness", "panel", "resolution"])),
            "mentions_performance": int(any(word in text for word in ["performance", "fast", "slow", "lag", "speed", "processor", "cpu", "ryzen", "intel"])),
            "mentions_heating": int(any(word in text for word in ["heat", "heating", "thermal", "temperature", "hot", "overheat"])),
            "mentions_build": int(any(word in text for word in ["build", "hinge", "body", "plastic", "metal", "durability"])),
            "mentions_price": int(any(word in text for word in ["price", "cost", "worth", "budget", "expensive", "cheap", "value"])),
            "mentions_gaming": int(any(word in text for word in ["gaming", "fps", "gpu", "rtx", "graphics", "game"])),
            "mentions_office": int(any(word in text for word in ["office", "student", "college", "coding", "work", "productivity", "study"]))
        }

    def search_reddit_posts(self, search_query, laptop_name, subreddit=None, sort="relevance", limit=25):
        print(f"\n[Reddit Search] Laptop: {laptop_name}")
        print(f"Query Used: {search_query}")
        if subreddit:
            print(f"Subreddit: r/{subreddit}")

        results = []
        after = None

        while len(results) < limit:
            base_url = "https://old.reddit.com/search.json"
            params = {
                "q": search_query,
                "sort": sort,
                "limit": 100,
                "type": "link"
            }

            if subreddit:
                params["restrict_sr"] = "on"
                params["q"] = f'subreddit:{subreddit} "{search_query}"'

            if after:
                params["after"] = after

            query_string = "&".join([f"{k}={quote(str(v))}" for k, v in params.items()])
            url = f"{base_url}?{query_string}"

            response = self.fetch_url(url)
            if not response:
                break

            try:
                data = json.loads(response)
                posts = data["data"]["children"]

                if not posts:
                    break

                for post in posts:
                    post_data = post["data"]

                    title = self.clean_text(post_data.get("title", ""))
                    selftext = self.clean_text(post_data.get("selftext", ""))
                    subreddit_name = post_data.get("subreddit", "")
                    score = post_data.get("score", 0)
                    created_utc = post_data.get("created_utc", None)
                    permalink = post_data.get("permalink", "")

                    combined_text = f"{title}. {selftext}".strip()
                    combined_text = self.clean_text(combined_text)

                    if self.count_words(combined_text) >= self.min_words:
                        normalized_text = self.normalize_text(combined_text)
                        topic_tags = self.tag_topics(combined_text)

                        results.append({
                            "laptop_name": laptop_name,
                            "query_used": search_query,
                            "subreddit": subreddit_name,
                            "title": title,
                            "selftext": selftext,
                            "combined_text": combined_text,
                            "normalized_text": normalized_text,
                            "score": score,
                            "created_utc": created_utc,
                            "permalink": f"https://www.reddit.com{permalink}" if permalink else "",
                            "text_length": self.count_words(combined_text),
                            "model_match_type": "exact" if search_query.lower() in normalized_text else "partial",
                            **topic_tags
                        })

                after = data["data"].get("after")
                if not after or len(results) >= limit:
                    break

                time.sleep(random.uniform(*self.delay))

            except Exception as e:
                print(f"JSON parse error: {e}")
                break

        return results[:limit]

    def scrape_model(self, laptop_name, subreddits=None, limit_per_query=25):
        all_results = []
        queries = self.get_query_variants(laptop_name)

        for q in queries:
            time.sleep(random.uniform(*self.delay))
            all_results.extend(
                self.search_reddit_posts(
                    search_query=q,
                    laptop_name=laptop_name,
                    subreddit=None,
                    sort="relevance",
                    limit=limit_per_query
                )
            )

            if subreddits:
                for subreddit in subreddits:
                    time.sleep(random.uniform(*self.delay))
                    all_results.extend(
                        self.search_reddit_posts(
                            search_query=q,
                            laptop_name=laptop_name,
                            subreddit=subreddit,
                            sort="relevance",
                            limit=limit_per_query
                        )
                    )

        if not all_results:
            self.stats["failed_models"] += 1
            print(f"No Reddit data found for: {laptop_name}")
            return None

        df = pd.DataFrame(all_results)

        # remove duplicates
        df = df.drop_duplicates(subset=["combined_text", "permalink"]).reset_index(drop=True)

        # remove weak rows
        df = df[df["text_length"] >= self.min_words].reset_index(drop=True)
        df = df[df["combined_text"].str.len() > 20].reset_index(drop=True)

        # order columns
        final_cols = [
            "laptop_name",
            "query_used",
            "subreddit",
            "title",
            "selftext",
            "combined_text",
            "normalized_text",
            "score",
            "created_utc",
            "permalink",
            "text_length",
            "model_match_type",
            "mentions_battery",
            "mentions_display",
            "mentions_performance",
            "mentions_heating",
            "mentions_build",
            "mentions_price",
            "mentions_gaming",
            "mentions_office"
        ]
        df = df[final_cols]

        file_name = self.safe_filename(laptop_name) + ".csv"
        file_path = os.path.join(self.output_dir, file_name)
        df.to_csv(file_path, index=False, encoding="utf-8-sig")

        self.stats["saved_rows"] += len(df)
        self.stats["saved_files"] += 1

        print(f"Saved {len(df)} rows -> {file_path}")
        return df

    def scrape_models(self, models, subreddits=None, limit_per_query=25):
        all_data = {}

        print(f"\n{'='*70}")
        print("STARTING REDDIT DATA COLLECTION")
        print(f"{'='*70}")

        for i, model in enumerate(models, 1):
            print(f"\n[{i}/{len(models)}] Scraping: {model}")
            try:
                df = self.scrape_model(
                    laptop_name=model,
                    subreddits=subreddits,
                    limit_per_query=limit_per_query
                )
                all_data[model] = df
            except Exception as e:
                self.stats["failed_models"] += 1
                print(f"Error scraping {model}: {e}")

        print(f"\n{'='*70}")
        print("FINAL REDDIT DATASET SUMMARY")
        print(f"{'='*70}")
        print(f"Saved rows: {self.stats['saved_rows']}")
        print(f"Saved files: {self.stats['saved_files']}")
        print(f"Failed models: {self.stats['failed_models']}")
        print(f"{'='*70}")

        return all_data

In [3]:
models = [
    "HP 15s-fq Series",
    "HP 15s-eq Series",
    "HP Pavilion 15-eg",
    "HP Pavilion Aero 13",
    "HP Victus 15",
    "HP Victus 16",
    "HP Victus 16 RTX 4050",
    "HP Omen 16",
    "HP Omen Transcend 14",
    "HP Omen 17"
]

In [4]:
subreddits = [
    "laptops",
    "SuggestALaptop",
    "GamingLaptops",
    "buildapc",
    "pcmasterrace",
    "HP"
]

In [5]:
scraper = RedditLaptopScraper(
    output_dir="reddit_dataset",
    min_words=5,
    delay=(1, 2)
)

all_reddit_data = scraper.scrape_models(
    models=models,
    subreddits=subreddits,
    limit_per_query=15
)


STARTING REDDIT DATA COLLECTION

[1/10] Scraping: HP 15s-fq Series

[Reddit Search] Laptop: HP 15s-fq Series
Query Used: HP 15s-fq Series

[Reddit Search] Laptop: HP 15s-fq Series
Query Used: HP 15s-fq Series
Subreddit: r/laptops

[Reddit Search] Laptop: HP 15s-fq Series
Query Used: HP 15s-fq Series
Subreddit: r/SuggestALaptop

[Reddit Search] Laptop: HP 15s-fq Series
Query Used: HP 15s-fq Series
Subreddit: r/GamingLaptops

[Reddit Search] Laptop: HP 15s-fq Series
Query Used: HP 15s-fq Series
Subreddit: r/buildapc

[Reddit Search] Laptop: HP 15s-fq Series
Query Used: HP 15s-fq Series
Subreddit: r/pcmasterrace

[Reddit Search] Laptop: HP 15s-fq Series
Query Used: HP 15s-fq Series
Subreddit: r/HP
Saved 15 rows -> reddit_dataset\HP_15s-fq_Series.csv

[2/10] Scraping: HP 15s-eq Series

[Reddit Search] Laptop: HP 15s-eq Series
Query Used: HP 15s-eq Series

[Reddit Search] Laptop: HP 15s-eq Series
Query Used: HP 15s-eq Series
Subreddit: r/laptops

[Reddit Search] Laptop: HP 15s-eq Series
Qu

In [6]:
os.listdir("reddit_dataset")

['all_reddit_cleaned.csv',
 'ASUS_ROG_Strix_G15.csv',
 'ASUS_ROG_Strix_Scar_15.csv',
 'ASUS_ROG_Zephyrus_G14.csv',
 'ASUS_TUF_A15_FA506.csv',
 'ASUS_TUF_F15_FX506.csv',
 'ASUS_TUF_F15_FX507.csv',
 'ASUS_VivoBook_15_X1502ZA.csv',
 'ASUS_VivoBook_16_X1605.csv',
 'ASUS_VivoBook_Go_15_E1504FA.csv',
 'ASUS_ZenBook_14_OLED.csv',
 'HP_15s-eq_Series.csv',
 'HP_15s-fq_Series.csv',
 'HP_Omen_16.csv',
 'HP_Omen_17.csv',
 'HP_Omen_Transcend_14.csv',
 'HP_Pavilion_15-eg.csv',
 'HP_Pavilion_Aero_13.csv',
 'HP_Victus_15.csv',
 'HP_Victus_16.csv',
 'HP_Victus_16_RTX_4050.csv']